In [1]:
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

# ==========================================
# 1. 영웅 님의 실제 윈도우 컴퓨터 절대 경로 설정
# ==========================================
# Path()를 사용하면 윈도우의 백슬래시(\) 경로 문제를 자동으로 해결해 줍니다.
BASE_DATA_DIR = Path(r"D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\animal_face_images")
TEST_DATA_DIR = Path(r"D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\test_images")

IMG_SIZE = 64

# 기획서 기준 동물상 매핑 (여우상이 기획서와 달라졌다면 추가/수정 가능) [cite: 24]
LABEL_MAP = {
    '강아지상': 0,
    '고양이상': 1,
    '토끼상': 2,
    '곰상': 3
    # 만약 여우상을 쓰실 거라면 아래처럼 주석을 풀고 추가해 주세요!
    # '여우상': 4 
}

# OpenCV 얼굴 정면 인식 모델 로드 [cite: 84]
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

print("✅ [설정 완료] 영웅 님 컴퓨터 경로 세팅이 정상적으로 끝났습니다.")
print(f" - 학습 데이터 경로: {BASE_DATA_DIR}")
print(f" - 판별용 데이터 경로: {TEST_DATA_DIR}\n")

# ==========================================
# 2. [학습 데이터용] 전처리 및 CSV 생성 파트
# ==========================================
flattened_data = []
labels = []

print("=== [1단계] 학습 데이터셋 전처리 가동 ===")
for folder_name, label_value in LABEL_MAP.items():
    # 폴더 경로를 윈도우 시스템에 맞게 결합
    folder_path = BASE_DATA_DIR / folder_name
    
    if not folder_path.exists():
        print(f"⚠️ 경고: '{folder_name}' 폴더를 찾을 수 없어 건너뜁니다.")
        continue
        
    print(f"▶ '{folder_name}' 처리 중... (Label: {label_value})")
    
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            img_path = folder_path / filename
            
            # 윈도우 한글 경로 깨짐 방지용 특수 로드 방식
            img_array = np.fromfile(str(img_path), np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
            
            if img is None: continue
                
            # 흑백 변환 및 얼굴 크롭 [cite: 84, 108]
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
            
            if len(faces) > 0:
                x, y, w, h = faces[0]
                cropped_face = gray[y:y+h, x:x+w]
            else:
                cropped_face = gray
                
            # 64x64 리사이즈 및 1차원 평탄화 [cite: 30, 84]
            resized_face = cv2.resize(cropped_face, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            flattened_data.append(resized_face.flatten())
            labels.append(label_value)

if len(flattened_data) > 0:
    pixel_columns = [f"pixel_{i}" for i in range(IMG_SIZE * IMG_SIZE)]
    df_train = pd.DataFrame(flattened_data, columns=pixel_columns)
    df_train['Label'] = labels
    
    # 주피터 파일과 같은 폴더에 CSV 저장 [cite: 110]
    df_train.to_csv("./animal_face_dataset.csv", index=False, encoding='utf-8-sig')
    print(f"💾 학습 데이터 구축 완료 -> 'animal_face_dataset.csv' 저장됨! (크기: {df_train.shape})\n")

# ==========================================
# 3. [판별(테스트)용] test_images 전처리 파트
# ==========================================
test_flattened_data = []
test_file_names = []

print("=== [2단계] 판별용(test_images) 데이터셋 전처리 가동 ===")
if not TEST_DATA_DIR.exists():
    print(f"💡 안내: '{TEST_DATA_DIR}' 폴더가 없습니다. 폴더를 생성해 주세요!")
else:
    for filename in os.listdir(TEST_DATA_DIR):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            img_path = TEST_DATA_DIR / filename
            
            img_array = np.fromfile(str(img_path), np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
            
            if img is None: continue
                
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
            
            if len(faces) > 0:
                x, y, w, h = faces[0]
                cropped_face = gray[y:y+h, x:x+w]
            else:
                cropped_face = gray
                
            resized_face = cv2.resize(cropped_face, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            test_flattened_data.append(resized_face.flatten())
            test_file_names.append(filename)

    if len(test_flattened_data) > 0:
        pixel_columns = [f"pixel_{i}" for i in range(IMG_SIZE * IMG_SIZE)]
        df_test = pd.DataFrame(test_flattened_data, columns=pixel_columns)
        df_test['Origin_Filename'] = test_file_names
        
        # 주피터 파일과 같은 폴더에 테스트용 CSV 저장 [cite: 178]
        df_test.to_csv("./unseen_test_dataset.csv", index=False, encoding='utf-8-sig')
        print(f"💾 판별용 데이터 구축 완료 -> 'unseen_test_dataset.csv' 저장됨! (개수: {df_test.shape[0]}개)")
    else:
        print("💡 안내: test_images 폴더에 넣은 사진이 없거나 읽을 수 없습니다.")

✅ [설정 완료] 영웅 님 컴퓨터 경로 세팅이 정상적으로 끝났습니다.
 - 학습 데이터 경로: D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\animal_face_images
 - 판별용 데이터 경로: D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\test_images

=== [1단계] 학습 데이터셋 전처리 가동 ===
▶ '강아지상' 처리 중... (Label: 0)
▶ '고양이상' 처리 중... (Label: 1)
⚠️ 경고: '토끼상' 폴더를 찾을 수 없어 건너뜁니다.
⚠️ 경고: '곰상' 폴더를 찾을 수 없어 건너뜁니다.
💾 학습 데이터 구축 완료 -> 'animal_face_dataset.csv' 저장됨! (크기: (66, 4097))

=== [2단계] 판별용(test_images) 데이터셋 전처리 가동 ===
💾 판별용 데이터 구축 완료 -> 'unseen_test_dataset.csv' 저장됨! (개수: 63개)
